
# TMDB Movie Data Analysis using Spark and PySpark

## 1. Project Introduction

This project implements a movie data analysis pipeline using Apache Spark and PySpark.

The main objectives are:
- Extract movie data from the TMDB API.
- Clean and transform movie datasets.
- Perform Exploratory Data Analysis (EDA).
- Calculate financial and popularity KPIs.
- Analyze movie franchises and directors.
- Visualize findings using Matplotlib.

## 2. Technology Stack

- Python
- Apache Spark
- PySpark
- TMDB API
- Pandas
- Matplotlib
- Jupyter Notebook



### 3. Spark Session Initialization

A SparkSession is the entry point for working with Spark DataFrames.

It allows us to:
- Read and process structured data.
- Execute Spark SQL queries.
- Perform distributed data transformations.
- Manage Spark application configurations.

The `getOrCreate()` method retrieves an existing SparkSession or creates a new one if none exists.


In [1]:

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TMDB Movie Data Analysis") \
    .getOrCreate()

print("Spark Version:", spark.version)
print("Spark Session Created Successfully!")


d:\AmaliTech\.pyspark-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark Version: 4.3.0
Spark Session Created Successfully!



## 4. API Data Extraction

In this section, we retrieve movie data from The Movie Database (TMDB) API.

We extract:
- Movie details
- Cast information
- Crew information

The extracted data will be stored in Python dictionaries before being converted into a Spark DataFrame.



## 4. Importing the Data Extraction Function

### 4.1 Import the Extraction Module

In this section, we import the `fetch_movie_data()` function from our
custom Python module `src/extraction.py`.

We separate reusable functions from the Jupyter Notebook to improve
code organization, maintainability, and reusability.

**Why do we do this?**

- Avoid writing the same code multiple times.
- Keep the notebook focused on the data analysis workflow.
- Make our extraction logic reusable in other Python scripts.
- Follow modular programming principles.

We use Python's `sys.path` to ensure that Python can locate our
project's `src` directory when importing the function.


In [3]:

import sys
from pathlib import Path

# Get the project root directory
project_root = Path.cwd().parent

# Add project root to Python's import path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import the extraction function
from src.extraction import fetch_movie_data

print("Extraction function imported successfully!")


Extraction function imported successfully!



### 4.2 Understanding the Import

- `Path.cwd()` returns the current working directory of the notebook.
- `.parent` moves one directory up to the project root.
- `sys.path` contains directories Python searches for modules.
- `from src.extraction import fetch_movie_data` imports the
  function defined in `extraction.py`.

The function is now available for use in our notebook without
reimplementing its extraction logic.



## 5. API Authentication

### 5.1 Load TMDB API Credentials

To access the TMDB API, we need an API Read Access Token.

We store the token in a `.env` file located in the root directory
of our project.

We use the `python-dotenv` library to load environment variables
into our Python environment.

**Why use environment variables?**

- Protect sensitive API credentials.
- Avoid hardcoding tokens in source code.
- Prevent credentials from being exposed in Git repositories.
- Make the project easier to configure across different environments.

Our `.env` file contains:

TMDB_API_TOKEN=your_api_read_access_token


In [5]:

import os
from dotenv import load_dotenv

# Load environment variables from the project root
load_dotenv(project_root / ".env")

# Retrieve the TMDB API token
TMDB_API_TOKEN = os.getenv("TMDB_READ_ACCESS_TOKEN")

# Validate the token
if not TMDB_API_TOKEN:
    raise ValueError(
        "TMDB API token not found. Please check your .env file."
    )

# Configure API request headers
headers = {
    "Authorization": f"Bearer {TMDB_API_TOKEN}",
    "accept": "application/json"
}

print("TMDB API authentication configured successfully!")


TMDB API authentication configured successfully!



### 5.2 Understanding the Authentication Code

- `load_dotenv()` loads variables from the `.env` file.
- `os.getenv()` retrieves the API token without exposing it directly.
- The `Authorization` header sends the token to TMDB.
- The `accept` header specifies that we expect a JSON response.
- The validation ensures that the token exists before making API requests.

The token is stored in the `headers` dictionary and will be passed
to our `fetch_movie_data()` extraction function.



### 5.3 Authentication Verification

We verify that the API credentials are configured correctly
before proceeding with data extraction.

We do not print the actual API token because it is a sensitive
credential.


In [8]:

print("Token loaded:", bool(TMDB_API_TOKEN))
print("Authorization header configured:", "Authorization" in headers)


Token loaded: True
Authorization header configured: True



## 6. Defining the Movie IDs

### 6.1 Select Movies for Analysis

We use the movie IDs provided in the project requirements
to identify the movies that we want to retrieve from TMDB.

Each movie has a unique identifier in the TMDB database.

We store these identifiers in a Python list called `movie_ids`.

This list will be passed to our `fetch_movie_data()` function
to retrieve movie details and credits.

**Note:** We include all 19 IDs provided in the assignment.
Some IDs may not return valid movie records.


In [9]:

movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397,
    420818, 24428, 168259, 99861, 284054, 12445,
    181808, 330457, 351286, 109445, 321612, 260513
]

print("Total movie IDs:", len(movie_ids))


Total movie IDs: 19



### 6.2 Validate the Movie ID List

Before extracting data, we perform a simple validation
to confirm that our list contains the expected number of IDs.

This helps ensure that we are using the correct input
for our extraction process.


In [10]:

assert len(movie_ids) == 19, "Expected exactly 19 movie IDs."

print("Movie ID validation successful.")


Movie ID validation successful.



## 7. Extracting Movie Data from TMDB

### 7.1 Execute the Data Extraction Function

We now execute the `fetch_movie_data()` function imported
from our `src/extraction.py` module.

The function iterates through the movie IDs and sends
HTTP GET requests to the TMDB API.

For each movie, we retrieve:
- Movie details, including title, budget, revenue, genres,
  release date, runtime, and popularity.
- Cast information, including actors and their character names.
- Crew information, including directors and other production members.

We use the `append_to_response=credits` parameter to retrieve
movie details and credits within a single API request.

The returned JSON responses are stored in the Python list
`movie_data`.

**Output:** A list of dictionaries containing the raw movie data
retrieved from TMDB.


In [11]:

movie_data = fetch_movie_data(movie_ids, headers)


Failed to fetch movie ID 0: 404 Client Error: Not Found for url: https://api.themoviedb.org/3/movie/0?append_to_response=credits&language=en-US
Fetched: 299534 - Avengers: Endgame
Fetched: 19995 - Avatar
Fetched: 140607 - Star Wars: The Force Awakens
Fetched: 299536 - Avengers: Infinity War
Fetched: 597 - Titanic
Fetched: 135397 - Jurassic World
Fetched: 420818 - The Lion King
Fetched: 24428 - The Avengers
Fetched: 168259 - Furious 7
Fetched: 99861 - Avengers: Age of Ultron
Fetched: 284054 - Black Panther
Fetched: 12445 - Harry Potter and the Deathly Hallows: Part 2
Fetched: 181808 - Star Wars: The Last Jedi
Fetched: 330457 - Frozen II
Fetched: 351286 - Jurassic World: Fallen Kingdom
Fetched: 109445 - Frozen
Fetched: 321612 - Beauty and the Beast
Fetched: 260513 - Incredibles 2

Extraction Summary
------------------
Successfully fetched: 18
Failed: 1



### 7.2 Evaluate the Extraction Results

After extraction, we evaluate the results to determine
how many movie records were successfully retrieved.

We compare the number of extracted records with the
number of requested movie IDs.

We also identify any failed requests that need further
investigation.

This validation helps us assess the completeness of
our extracted dataset before moving to data inspection
and transformation.


In [12]:

print("Requested movie IDs:", len(movie_ids))
print("Successfully extracted movies:", len(movie_data))
print("Extraction completeness:", 
      round(len(movie_data) / len(movie_ids) * 100, 2), "%")


Requested movie IDs: 19
Successfully extracted movies: 18
Extraction completeness: 94.74 %



### 7.3 Inspect the Raw Extracted Data

We inspect one movie record to understand the structure
of the JSON response returned by TMDB.

The API response contains several nested objects and arrays.

Understanding this structure is important because we will
later extract and transform these fields into a structured
Spark DataFrame.

We begin by inspecting the available keys in the first
successfully extracted movie record.


In [13]:

movie_data[0].keys()


dict_keys(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'credits'])


### 7.4 Inspect Movie Credits

The `credits` field contains information about the movie's
cast and crew.

We inspect its structure to understand how actor and director
information is organized.

The cast will later be used for actor-based movie searches,
while the crew data will be used to identify directors.


In [14]:

movie_data[0]["credits"].keys()


dict_keys(['cast', 'crew'])


### 7.5 Inspect the Raw JSON Response

We display the complete JSON response of the first movie
to explore the nested data structures.

This inspection helps us identify:
- Fields that require extraction.
- Nested JSON objects.
- Arrays containing multiple values.
- Fields that may contain missing information.

We use this information to design our data cleaning
and transformation functions.


In [15]:

import json

print(json.dumps(movie_data[0], indent=2))


{
  "adult": false,
  "backdrop_path": "/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg",
  "belongs_to_collection": {
    "id": 86311,
    "name": "The Avengers Collection",
    "poster_path": "/yFSIUVTCvgYrpalUktulvk3Gi5Y.jpg",
    "backdrop_path": "/2UNUv4NJdC36E5myDHACBJ99EwL.jpg"
  },
  "budget": 356000000,
  "genres": [
    {
      "id": 12,
      "name": "Adventure"
    },
    {
      "id": 878,
      "name": "Science Fiction"
    },
    {
      "id": 28,
      "name": "Action"
    }
  ],
  "homepage": "https://www.marvel.com/movies/avengers-endgame",
  "id": 299534,
  "imdb_id": "tt4154796",
  "origin_country": [
    "US"
  ],
  "original_language": "en",
  "original_title": "Avengers: Endgame",
  "overview": "After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no m


## 8. Saving Raw Data — Bronze Layer

### 8.1 Preserve the Raw API Responses

Before performing data cleaning and transformation, we save
the raw movie data retrieved from the TMDB API.

We store the extracted records in JSON format inside the
`data/raw/` directory.

This represents the Bronze Layer of our data pipeline.

The raw data remains unchanged so that we can reuse it
for future processing, debugging, and data recovery.

**Important:** We do not perform cleaning or remove any
fields from the raw dataset at this stage.


In [16]:


import json

# Create the raw data directory if it does not exist
raw_data_path = project_root / "data" / "raw"
raw_data_path.mkdir(parents=True, exist_ok=True)

# Define the output file
raw_file = raw_data_path / "tmdb_movies_raw.json"

# Save raw movie data
with open(raw_file, "w", encoding="utf-8") as file:
    json.dump(movie_data, file, indent=4, ensure_ascii=False)

print(f"Raw data saved successfully: {raw_file}")
print(f"Total records saved: {len(movie_data)}")



Raw data saved successfully: d:\AmaliTech\tmdb-spark-analysis\data\raw\tmdb_movies_raw.json
Total records saved: 18



### 8.2 Verify the Raw Data File

After saving the extracted data, we verify that the file
exists and contains the expected number of records.

This validation ensures that our Bronze Layer has been
successfully created before proceeding to data cleaning.


In [17]:


# Verify the saved raw dataset
with open(raw_file, "r", encoding="utf-8") as file:
    raw_data_check = json.load(file)

print("File exists:", raw_file.exists())
print("Records in saved file:", len(raw_data_check))



File exists: True
Records in saved file: 18



## 9. Data Cleaning and Preprocessing — Silver Layer

### 9.1 Load Raw JSON Data into a Spark DataFrame

In this section, we load the raw movie data saved in the
Bronze Layer into a Spark DataFrame.

We use Spark's JSON reader to read the JSON file.

The `multiline` option is set to `true` because our JSON
file contains formatted JSON data spanning multiple lines.

Spark automatically infers the schema from the JSON data.

At this stage, we do not modify the data. We only load
and inspect its structure before beginning the cleaning process.


In [18]:

# Load raw JSON data into Spark DataFrame

raw_df = spark.read \
    .option("multiline", "true") \
    .json(str(raw_file))

print("Raw DataFrame loaded successfully!")


Raw DataFrame loaded successfully!



### 9.2 Inspect the Raw DataFrame Schema

We use the `printSchema()` method to examine the structure
and data types of the columns in our Spark DataFrame.

This is important because the TMDB API returns nested JSON
data containing arrays and structures.

For example:
- `genres` is an array of genre objects.
- `production_companies` is an array of company objects.
- `credits` is a nested structure containing cast and crew.
- `belongs_to_collection` is a nested collection object.

Understanding the schema helps us determine which columns
need to be extracted, transformed, or converted.


In [19]:

raw_df.printSchema()


root
 |-- adult: boolean (nullable = true)
 |-- backdrop_path: string (nullable = true)
 |-- belongs_to_collection: struct (nullable = true)
 |    |-- backdrop_path: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- poster_path: string (nullable = true)
 |-- budget: long (nullable = true)
 |-- credits: struct (nullable = true)
 |    |-- cast: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- adult: boolean (nullable = true)
 |    |    |    |-- cast_id: long (nullable = true)
 |    |    |    |-- character: string (nullable = true)
 |    |    |    |-- credit_id: string (nullable = true)
 |    |    |    |-- gender: long (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- known_for_department: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- order: long (nullable = true)
 |    |    |    |-- original_nam


### 9.3 Inspect the Number of Records and Columns

We use `count()` to determine the number of movie records
in the DataFrame.

We use `columns` to retrieve the names of all columns.

These operations provide an initial overview of the dataset
and help us verify that the raw data was loaded correctly.


In [20]:

print("Number of records:", raw_df.count())
print("Number of columns:", len(raw_df.columns))

print("\nColumn names:")
print(raw_df.columns)


Number of records: 18
Number of columns: 28

Column names:
['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'credits', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']



### 9.4 Display Sample Records

We use the `show()` method to display a sample of the
raw movie data.

The `truncate=False` parameter prevents Spark from
truncating long column values.

This allows us to visually inspect the data before
implementing the cleaning operations.


In [21]:

raw_df.show(5, truncate=False)


+-----+--------------------------------+----------------------------------------------------------------------------------------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


### 10.1 Remove Irrelevant Columns

The raw TMDB dataset contains columns that are not required
for our analysis.

We remove the following columns:

- `adult`: Adult content indicator.
- `imdb_id`: IMDb identifier.
- `original_title`: Original movie title.
- `video`: Video availability indicator.
- `homepage`: Movie website URL.
- `softcore`: An additional field not required by our project.

We use the Spark `drop()` method to remove these columns.

This reduces the number of columns we need to process
during subsequent transformations.

We preserve the original raw JSON data in the Bronze Layer.
Therefore, these removals only affect our working DataFrame.



### 10.2 Import and Execute the Cleaning Function

We import the `drop_irrelevant_columns()` function from
our `src/cleaning.py` module.

We then apply the function to our raw Spark DataFrame
to create a new DataFrame called `df`.

The variable `df` represents our working DataFrame
for the Silver Layer.

We retain `raw_df` as the original in-memory DataFrame
and preserve the original raw JSON file in the Bronze Layer.


In [22]:

from src.cleaning import drop_irrelevant_columns

df = drop_irrelevant_columns(raw_df)

print("Irrelevant columns removed successfully!")
print("Remaining columns:", len(df.columns))


Irrelevant columns removed successfully!
Remaining columns: 22



### 10.3 Verify the Column Removal

We inspect the updated DataFrame schema to confirm
that the irrelevant columns have been removed.

We also display a sample of the remaining data
to verify that the cleaning operation did not
remove the movie records themselves.


In [23]:

df.printSchema()

print("Number of records:", df.count())

df.show(5, truncate=False)


root
 |-- backdrop_path: string (nullable = true)
 |-- belongs_to_collection: struct (nullable = true)
 |    |-- backdrop_path: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- poster_path: string (nullable = true)
 |-- budget: long (nullable = true)
 |-- credits: struct (nullable = true)
 |    |-- cast: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- adult: boolean (nullable = true)
 |    |    |    |-- cast_id: long (nullable = true)
 |    |    |    |-- character: string (nullable = true)
 |    |    |    |-- credit_id: string (nullable = true)
 |    |    |    |-- gender: long (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- known_for_department: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- order: long (nullable = true)
 |    |    |    |-- original_name: string (nullable = true)
 |    |   


### 11.1 Extract Movie Collection Name

The `belongs_to_collection` column is a nested structure
containing information about a movie's collection.

For example, a movie may belong to a collection such as
"Avatar Collection" or "The Avengers Collection".

We extract the `name` field from the nested structure
using Spark's column access syntax.

Movies that do not belong to a collection will have
a null collection name.

This allows us to distinguish franchise movies from
standalone movies during our later analysis.



### 11.2 Extract and Combine Genre Names

The `genres` column contains an array of genre objects.
Each object contains a genre ID and a genre name.

We use Spark's `transform()` function to extract the
name from each genre object.

We then use `concat_ws("|", ...)` to combine the
genre names into a single string separated by `|`.

For example:

[{"id": 28, "name": "Action"},
 {"id": 878, "name": "Science Fiction"}]

Becomes:

"Action|Science Fiction"

This format makes it easier to inspect and filter
movies by their genres.



### 11.3 Extract and Combine Spoken Languages

The `spoken_languages` column contains an array of
language objects.

Each language object includes:
- `english_name`: English representation of the language.
- `iso_639_1`: Language code.
- `name`: Language name returned by TMDB.

We extract the `name` field from each object and
combine the values using the `|` separator.

This creates a simple string representation of
the spoken languages for each movie.



### 11.4 Extract and Combine Production Countries

The `production_countries` column contains an array
of country objects.

Each object includes a country name and its ISO code.

We extract the `name` field from each country object
and combine the country names using `|`.

This transformation simplifies the representation
of movie production locations.



### 11.5 Extract and Combine Production Company Names

The `production_companies` column contains an array
of production company objects.

Each object includes information such as company ID,
company name, logo path, and country of origin.

We extract the `name` field from each company object
and combine the company names using `|`.

This transformation allows us to analyze movie production
companies without working with nested structures.



### 11.6 Extract Cast Members and Calculate Cast Size

The `credits.cast` column contains an array of cast
members who participated in each movie.

Each cast object includes the actor's name, character,
cast order, and other information.

We perform two transformations:

1. Extract the actor names and combine them using `|`.
2. Calculate the number of cast members using Spark's `size()` function.

The resulting columns are:
- `cast`: A string containing the cast member names.
- `cast_size`: The number of cast members.

We use `coalesce()` to replace null cast arrays with
an empty array before calculating their size.



### 11.7 Extract Director Name and Calculate Crew Size

The `credits.crew` column contains an array of crew
members involved in movie production.

Crew members have different jobs, such as:
- Director
- Producer
- Editor
- Writer
- Cinematographer

We identify the director by filtering the crew array
where the `job` field equals `"Director"`.

We extract the director's name and store it in
the `director` column.

We also calculate the total number of crew members
using Spark's `size()` function.

If multiple directors are returned, we select the first
director in the filtered array.



## 12. Executing Nested Data Transformations

### 12.1 Apply the Extraction Functions

We import the extraction functions from `src/cleaning.py`
and apply them sequentially to our working DataFrame.

The transformations extract:
- Collection name
- Genre names
- Spoken languages
- Production countries
- Production companies
- Cast members and cast size
- Director name and crew size

We use the returned DataFrame from each function
as the input for the next transformation.

This approach allows us to build our cleaned dataset
incrementally while keeping the transformation logic
organized in reusable Python functions.


In [27]:
import importlib
import src.cleaning

importlib.reload(src.cleaning)

from src.cleaning import (
    extract_collection_name,
    extract_genres,
    extract_spoken_languages,
    extract_production_countries,
    extract_production_companies,
    extract_cast_and_size,
    extract_director_and_crew_size
)

df = extract_collection_name(df)
df = extract_genres(df)
df = extract_spoken_languages(df)
df = extract_production_countries(df)
df = extract_production_companies(df)
df = extract_cast_and_size(df)
df = extract_director_and_crew_size(df)

print("Nested data transformations completed successfully!")


Nested data transformations completed successfully!



### 12.2 Inspect the Transformed Columns

We display the newly extracted columns to verify
that the transformations were performed correctly.

We expect nested arrays and structures to be replaced
with simple string or numeric columns.

We also inspect sample values to identify possible
missing values or unexpected results.


In [28]:

df.select(
    "title",
    "belongs_to_collection",
    "genres",
    "spoken_languages",
    "production_countries",
    "production_companies",
    "cast",
    "cast_size",
    "director",
    "crew_size"
).show(5, truncate=False)


+----------------------------+-----------------------+--------------------------------+-------------------------------------------------+---------------------------------------+------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


### 12.3 Inspect the Updated Schema

We use `printSchema()` to verify that the transformed
columns have the expected data types.

The extracted name columns should now be strings,
while `cast_size` and `crew_size` should be numeric.

The original `credits` column still exists at this stage.
We will remove it after confirming that the required
information has been extracted successfully.


In [29]:

df.printSchema()


root
 |-- backdrop_path: string (nullable = true)
 |-- belongs_to_collection: string (nullable = true)
 |-- budget: long (nullable = true)
 |-- credits: struct (nullable = true)
 |    |-- cast: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- adult: boolean (nullable = true)
 |    |    |    |-- cast_id: long (nullable = true)
 |    |    |    |-- character: string (nullable = true)
 |    |    |    |-- credit_id: string (nullable = true)
 |    |    |    |-- gender: long (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- known_for_department: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- order: long (nullable = true)
 |    |    |    |-- original_name: string (nullable = true)
 |    |    |    |-- popularity: double (nullable = true)
 |    |    |    |-- profile_path: string (nullable = true)
 |    |-- crew: array (nullable = true)
 |    |    |-- element: struct


## 13. Removing Unnecessary Nested Columns

### 13.1 Drop the Credits Column

The `credits` column contains nested cast and crew
information from the TMDB API.

We have already extracted the required information
into the following columns:

- `cast`
- `cast_size`
- `director`
- `crew_size`

Therefore, the original `credits` column is no longer
necessary for our current analysis.

We use the Spark `drop()` method to remove it.

This simplifies our DataFrame and eliminates redundant
nested information from the Silver Layer.



### 13.2 Apply the Credits Removal Function

We import the `drop_credits_column()` function from
our cleaning module and apply it to the working DataFrame.

The resulting DataFrame contains the extracted cast,
director, and crew information without the original
nested credits structure.


In [30]:
import importlib
import src.cleaning

importlib.reload(src.cleaning)

from src.cleaning import drop_credits_column

df = drop_credits_column(df)

print("Credits column removed successfully!")


Credits column removed successfully!



### 13.3 Verify the Updated DataFrame

We inspect the updated schema to confirm that
the `credits` column has been removed.

We also verify that our extracted columns remain
available for further analysis.


In [31]:

df.printSchema()

print("Number of columns:", len(df.columns))
print("Credits column exists:", "credits" in df.columns)


root
 |-- backdrop_path: string (nullable = true)
 |-- belongs_to_collection: string (nullable = true)
 |-- budget: long (nullable = true)
 |-- genres: string (nullable = false)
 |-- id: long (nullable = true)
 |-- origin_country: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- original_language: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- poster_path: string (nullable = true)
 |-- production_companies: string (nullable = false)
 |-- production_countries: string (nullable = false)
 |-- release_date: string (nullable = true)
 |-- revenue: long (nullable = true)
 |-- runtime: long (nullable = true)
 |-- spoken_languages: string (nullable = false)
 |-- status: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- title: string (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- vote_count: long (nullable = true)
 |-- cast: string (nullable = false)
 |-- cast_size:

### Convert Data Types

In this step, we convert columns into appropriate data types to support accurate calculations and analysis. For example, release dates are converted into date values, while budget and revenue are converted into numeric values.

In [32]:
import importlib
import src.cleaning

importlib.reload(src.cleaning)
from src.cleaning import convert_data_types

df = convert_data_types(df)

df.printSchema()

root
 |-- backdrop_path: string (nullable = true)
 |-- belongs_to_collection: string (nullable = true)
 |-- budget: long (nullable = true)
 |-- genres: string (nullable = false)
 |-- id: long (nullable = true)
 |-- origin_country: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- original_language: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- poster_path: string (nullable = true)
 |-- production_companies: string (nullable = false)
 |-- production_countries: string (nullable = false)
 |-- release_date: date (nullable = true)
 |-- revenue: long (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- spoken_languages: string (nullable = false)
 |-- status: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- title: string (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- vote_count: integer (nullable = true)
 |-- cast: string (nullable = false)
 |-- cast_s

### 13.4 Identify Missing Values

In this step, we identify missing values in each column of the cleaned DataFrame. This helps us determine which columns require further cleaning before performing analysis.

In [33]:
import importlib
import src.cleaning

importlib.reload(src.cleaning)

from src.cleaning import missing_value_report

missing_df = missing_value_report(df)

missing_df.show(truncate=False)

+-------------+---------------------+------+------+---+--------------+-----------------+--------+----------+-----------+--------------------+--------------------+------------+-------+-------+----------------+------+-------+-----+------------+----------+----+---------+--------+---------+
|backdrop_path|belongs_to_collection|budget|genres|id |origin_country|original_language|overview|popularity|poster_path|production_companies|production_countries|release_date|revenue|runtime|spoken_languages|status|tagline|title|vote_average|vote_count|cast|cast_size|director|crew_size|
+-------------+---------------------+------+------+---+--------------+-----------------+--------+----------+-----------+--------------------+--------------------+------------+-------+-------+----------------+------+-------+-----+------------+----------+----+---------+--------+---------+
|0            |2                    |0     |0     |0  |0             |0                |0       |0         |0          |0               

### 13.5 Handle Missing Collection Values

Two movies do not belong to any collection. Instead of removing these records, we replace their missing collection names with `"No Collection"`. This preserves all movie records while making the collection field easier to analyze.

In [34]:
import importlib
import src.cleaning

importlib.reload(src.cleaning)
from src.cleaning import handle_missing_values

df = handle_missing_values(df)

df.select(
    "title",
    "belongs_to_collection"
).show(truncate=False)

+--------------------------------------------+-----------------------------------+
|title                                       |belongs_to_collection              |
+--------------------------------------------+-----------------------------------+
|Avengers: Endgame                           |The Avengers Collection            |
|Avatar                                      |Avatar Collection                  |
|Star Wars: The Force Awakens                |Star Wars Collection               |
|Avengers: Infinity War                      |The Avengers Collection            |
|Titanic                                     |No Collection                      |
|Jurassic World                              |Jurassic Park Collection           |
|The Lion King                               |The Lion King (Reboot) Collection  |
|The Avengers                                |The Avengers Collection            |
|Furious 7                                   |The Fast and the Furious Collection|
|Ave

### 13.6 Check for Unrealistic Values

In this step, we inspect numeric columns to identify values that fall outside their expected ranges. This helps prevent invalid data from affecting our calculations and analysis.

In [35]:
import importlib
import src.cleaning

importlib.reload(src.cleaning)

from src.cleaning import check_unrealistic_values

unrealistic_report = check_unrealistic_values(df)

unrealistic_report.show()

+---------------+----------------+---------------+--------------------+-------------------+-------------------+
|negative_budget|negative_revenue|invalid_runtime|invalid_vote_average|negative_vote_count|negative_popularity|
+---------------+----------------+---------------+--------------------+-------------------+-------------------+
|              0|               0|              0|                   0|                  0|                  0|
+---------------+----------------+---------------+--------------------+-------------------+-------------------+



### 13.7 Check for Duplicate Movie Records

In this step, we check for duplicate movie records using the TMDB movie ID as the unique identifier. This ensures that each movie is represented only once in our dataset.

In [37]:
import importlib
import src.cleaning

importlib.reload(src.cleaning)

from src.cleaning import duplicate_movie_report

duplicate_report = duplicate_movie_report(df)

duplicate_report.show()

+-------------------+-----------------------+
|duplicate_id_groups|extra_duplicate_records|
+-------------------+-----------------------+
|                  0|                      0|
+-------------------+-----------------------+



### 13.8 Remove Duplicate Movie Records

In this step, we remove duplicate records using the TMDB movie ID as the unique identifier. This ensures that each movie appears only once in the final dataset.

In [38]:
import importlib
import src.cleaning

importlib.reload(src.cleaning)
from src.cleaning import remove_duplicate_movies

df = remove_duplicate_movies(df)

print("Total movies after removing duplicates:", df.count())

Total movies after removing duplicates: 18


### 13.9 Final Verification of the Cleaned DataFrame

In this final cleaning step, we verify the number of movie records, confirm that movie IDs are unique, inspect the number of columns, and review the final schema. This ensures that the dataset is ready for analysis.

In [39]:
import importlib
import src.cleaning

importlib.reload(src.cleaning)

from src.cleaning import final_data_quality_report

# Check total movies and unique IDs
final_data_quality_report(df).show()

# Check number of columns
print("Total columns:", len(df.columns))

# Inspect final schema
df.printSchema()

# Preview cleaned records
df.show(5, truncate=False)

+------------+----------------+
|total_movies|unique_movie_ids|
+------------+----------------+
|          18|              18|
+------------+----------------+

Total columns: 25
root
 |-- backdrop_path: string (nullable = true)
 |-- belongs_to_collection: string (nullable = true)
 |-- budget: long (nullable = true)
 |-- genres: string (nullable = false)
 |-- id: long (nullable = true)
 |-- origin_country: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- original_language: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- poster_path: string (nullable = true)
 |-- production_companies: string (nullable = false)
 |-- production_countries: string (nullable = false)
 |-- release_date: date (nullable = true)
 |-- revenue: long (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- spoken_languages: string (nullable = false)
 |-- status: string (nullable = true)
 |-- tagline: string (nullable =

### 13.10 Save the Cleaned DataFrame

After completing the cleaning and verification steps, we save the processed movie dataset in Parquet format under `data/processed/`. This preserves the data types and allows us to reuse the cleaned dataset in subsequent analysis without repeating the extraction and cleaning processes.

In [42]:
# Define the processed data directory
processed_dir = project_root / "data" / "processed"

# Create the directory if it does not exist
processed_dir.mkdir(parents=True, exist_ok=True)

# Define the output path
processed_file = processed_dir / "tmdb_movies_cleaned"

# Save the cleaned DataFrame as Parquet
df.toPandas().to_parquet(str(processed_file) + ".parquet", index=False)

print("Cleaned dataset saved successfully!")
print("Location:", processed_file)

d:\AmaliTech\.pyspark-env\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Cleaned dataset saved successfully!
Location: d:\AmaliTech\tmdb-spark-analysis\data\processed\tmdb_movies_cleaned


### 13.11 Verify the Saved Processed Dataset

We reload the saved Parquet dataset to confirm that the data was stored successfully and that the records and schema were preserved.

In [ ]:
import pandas as pd

pdf = pd.read_parquet(str(processed_file) + ".parquet")
processed_df = spark.createDataFrame(pdf)

print("Saved records:", processed_df.count())
print("Saved columns:", len(processed_df.columns))